# Sprint 2 — Pares entrada/alvo: o deslocamento de uma posição

**O que a Sprint 2 recebe da Frente A:** o `BPETokenizer` (`src/tokenizer/bpe.py`), que
converte o corpus "The Verdict" em uma sequência de Token IDs sobre o vocabulário fechado
do GPT-2 (50 257 entradas) e sabe desfazer essa conversão com `decode`. A Frente A entrega
texto virando IDs; a partir daqui os IDs são o único material de trabalho — não se volta a
olhar caracteres ou palavras soltas.

**O que esta etapa produz para a Sprint 3:** pares `(entrada, alvo)` recortados dessa
sequência de IDs, onde o alvo é a entrada deslocada uma posição à frente. É esse par que a
Self-Attention da Sprint 3 vai consumir: a entrada é o que o modelo enxerga, o alvo é o que
ele precisa aprender a prever no passo seguinte.

Este notebook existe para tornar esse deslocamento visível **antes** de qualquer
abstração — nada daqui é reaproveitado como código de `src/`. A versão reutilizável
(`GPTDataset`, com `__len__`/`__getitem__` e integração com `torch.utils.data.DataLoader`)
vive em `src/embeddings/dataset.py` e só faz sentido depois de ver o mecanismo na mão.

In [1]:
import sys
from pathlib import Path

# Convenção do projeto: `jupyter notebook` é aberto na raiz do repositório (ver README),
# então o kernel deste notebook roda com cwd = notebooks/sprint02/. Dois níveis acima é a raiz.
PROJECT_ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.corpus import load_verdict
from src.tokenizer import BPETokenizer

corpus = load_verdict()
bpe = BPETokenizer()
ids = bpe.encode(corpus)

print(f"Corpus 'The Verdict': {len(corpus)} caracteres")
print(f"Token IDs (BPE):      {len(ids)}")
print(f"Primeiros 20 IDs:     {ids[:20]}")
print(f"Decode dos 20:        {bpe.decode(ids[:20])!r}")

Corpus 'The Verdict': 20479 caracteres
Token IDs (BPE):      5145
Primeiros 20 IDs:     [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438]
Decode dos 20:        'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--'


## Passo 1 — `context_length=4`, `stride=4` (sem sobreposição)

Cada amostra é uma janela de `context_length` Token IDs. O alvo é a mesma janela deslocada
uma posição: `alvo[i] = entrada[i + 1]`, e o último ID do alvo é o próximo token do corpus
que a janela de entrada ainda não tinha visto.

Com `stride == context_length`, a próxima janela começa exatamente onde a anterior parou —
nenhum Token ID é reaproveitado entre amostras consecutivas.

In [2]:
context_length = 4
stride = 4
num_amostras_exibidas = 5

print(f"context_length={context_length}, stride={stride}  (overlap esperado = {context_length - stride} tokens)\n")

for start in range(0, num_amostras_exibidas * stride, stride):
    end = start + context_length
    entrada_ids = ids[start:end]
    alvo_ids = ids[start + 1:end + 1]

    entrada_texto = bpe.decode(entrada_ids)
    alvo_texto = bpe.decode(alvo_ids)

    print(f"amostra start={start}")
    print(f"  entrada IDs   : {entrada_ids}")
    print(f"  alvo    IDs   : {alvo_ids}")
    print(f"  entrada texto : {entrada_texto!r:<28} | alvo texto : {alvo_texto!r}")
    print()

context_length=4, stride=4  (overlap esperado = 0 tokens)

amostra start=0
  entrada IDs   : [40, 367, 2885, 1464]
  alvo    IDs   : [367, 2885, 1464, 1807]
  entrada texto : 'I HAD always'               | alvo texto : ' HAD always thought'

amostra start=4
  entrada IDs   : [1807, 3619, 402, 271]
  alvo    IDs   : [3619, 402, 271, 10899]
  entrada texto : ' thought Jack Gis'          | alvo texto : ' Jack Gisburn'

amostra start=8
  entrada IDs   : [10899, 2138, 257, 7026]
  alvo    IDs   : [2138, 257, 7026, 15632]
  entrada texto : 'burn rather a cheap'        | alvo texto : ' rather a cheap genius'

amostra start=12
  entrada IDs   : [15632, 438, 2016, 257]
  alvo    IDs   : [438, 2016, 257, 922]
  entrada texto : ' genius--though a'          | alvo texto : '--though a good'

amostra start=16
  entrada IDs   : [922, 5891, 1576, 438]
  alvo    IDs   : [5891, 1576, 438, 568]
  entrada texto : ' good fellow enough--'      | alvo texto : ' fellow enough--so'



## Passo 2 — `context_length=4`, `stride=2` (com sobreposição)

Agora o stride é menor que o contexto. A próxima janela começa antes do fim da anterior,
então parte dos Token IDs aparece em mais de uma amostra — o texto "avança" mais devagar
que o tamanho da janela.

In [3]:
context_length = 4
stride = 2
num_amostras_exibidas = 5

print(f"context_length={context_length}, stride={stride}  (overlap esperado = {context_length - stride} tokens)\n")

janelas = []
for start in range(0, num_amostras_exibidas * stride, stride):
    end = start + context_length
    entrada_ids = ids[start:end]
    alvo_ids = ids[start + 1:end + 1]
    janelas.append((start, entrada_ids, alvo_ids))

    entrada_texto = bpe.decode(entrada_ids)
    alvo_texto = bpe.decode(alvo_ids)

    print(f"amostra start={start}")
    print(f"  entrada IDs   : {entrada_ids}")
    print(f"  alvo    IDs   : {alvo_ids}")
    print(f"  entrada texto : {entrada_texto!r:<28} | alvo texto : {alvo_texto!r}")
    print()

print("Sobreposição entre amostras consecutivas (IDs de entrada em comum):")
for (start_a, entrada_a, _), (start_b, entrada_b, _) in zip(janelas, janelas[1:]):
    comuns = [token_id for token_id in entrada_a if token_id in entrada_b]
    print(f"  start={start_a} vs start={start_b}: {comuns}  ({len(comuns)}/{context_length} tokens em comum)")

context_length=4, stride=2  (overlap esperado = 2 tokens)

amostra start=0
  entrada IDs   : [40, 367, 2885, 1464]
  alvo    IDs   : [367, 2885, 1464, 1807]
  entrada texto : 'I HAD always'               | alvo texto : ' HAD always thought'

amostra start=2
  entrada IDs   : [2885, 1464, 1807, 3619]
  alvo    IDs   : [1464, 1807, 3619, 402]
  entrada texto : 'AD always thought Jack'     | alvo texto : ' always thought Jack G'

amostra start=4
  entrada IDs   : [1807, 3619, 402, 271]
  alvo    IDs   : [3619, 402, 271, 10899]
  entrada texto : ' thought Jack Gis'          | alvo texto : ' Jack Gisburn'

amostra start=6
  entrada IDs   : [402, 271, 10899, 2138]
  alvo    IDs   : [271, 10899, 2138, 257]
  entrada texto : ' Gisburn rather'            | alvo texto : 'isburn rather a'

amostra start=8
  entrada IDs   : [10899, 2138, 257, 7026]
  alvo    IDs   : [2138, 257, 7026, 15632]
  entrada texto : 'burn rather a cheap'        | alvo texto : ' rather a cheap genius'

Sobreposição entre a

## Observação

Em toda amostra, o alvo é a entrada deslocada uma posição — `entrada[1:] == alvo[:-1]`, e
o último ID do alvo é o único Token ID novo que a janela ainda não continha. Isso não muda
com o stride: o deslocamento é sempre de uma posição, é uma propriedade de como o par é
recortado, não do tamanho do passo entre janelas.

O que o stride controla é a sobreposição **entre amostras consecutivas**, não dentro de uma
amostra:

- `stride == context_length` (Passo 1): cada Token ID pertence a exatamente uma janela de
  entrada. Zero sobreposição, uma passada pelo corpus gera o menor número possível de
  amostras.
- `stride < context_length` (Passo 2): janelas consecutivas compartilham
  `context_length - stride` Token IDs. Mais amostras a partir do mesmo corpus, à custa de
  reapresentar o mesmo trecho de texto em contextos diferentes.